# Projeto Python IA: Inteligência Artificial e Previsões

### Case: Score de Crédito dos Clientes

Você foi contratado por um banco para conseguir definir o score de crédito dos clientes. Você precisa analisar todos os clientes do banco e, com base nessa análise, criar um modelo que consiga ler as informações do cliente e dizer automaticamente o score de crédito dele: Ruim, Ok, Bom

Arquivos da aula: https://drive.google.com/drive/folders/1FbDqVq4XLvU85VBlVIMJ73p9oOu6u2-J?usp=drive_link

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# ---------- CONFIG ----------
ARQ_TREINO = "Banco_de_Fardos.xlsx"
ARQ_NOVOS = "Novos_Fardos.xlsx"
ARQ_SAIDA = "PlanilhaAtualizada.xlsx"

# Colunas que você disse que existem
COLS_ESP = ["EAN", "Material", "Nome CONC", "Conversao", "Fator"]

# ---------- Funções utilitárias ----------
def safe_read_excel(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    return pd.read_excel(path)

def normalize_colnames(df):
    # Remove espaços laterais
    df = df.copy()
    df.columns = df.columns.str.strip()
    return df

# ---------- 1) Ler e preparar base de treino ----------
tabela = safe_read_excel(ARQ_TREINO)
tabela = normalize_colnames(tabela)

# Verificar colunas essenciais (se faltar, avisar)
faltantes = [c for c in COLS_ESP if c not in tabela.columns]
if faltantes:
    print(f"Aviso: colunas esperadas faltando no treino: {faltantes}")
    # Você pode optar por: raise Exception(...) — aqui vamos continuar, mas cuidado
    for c in faltantes:
        tabela[c] = np.nan

# Conversao: manter como string e remover linhas sem target
tabela = tabela.dropna(subset=["Conversao"])
tabela["Conversao"] = tabela["Conversao"].astype(str)

# Fator -> numérico (coerce) e preencher NaN
tabela["Fator"] = pd.to_numeric(tabela["Fator"], errors="coerce").fillna(0.0)

# ---------- 2) Codificar colunas categóricas (com segurança) ----------
codificadores = {}
# Forçar a tratar EAN/Material/Nome CONC como string antes de treinar
for coluna in ["EAN", "Material", "Nome CONC"]:
    if coluna in tabela.columns:
        # preencher NaN com texto padrão para garantir consistência
        tabela[coluna] = tabela[coluna].astype(str).fillna("Desconhecido")
        le = LabelEncoder()
        le.fit(tabela[coluna])                # treina com as strings
        tabela[coluna] = le.transform(tabela[coluna])
        codificadores[coluna] = le

# ---------- 3) Preparar X e y ----------
y1 = tabela["Conversao"]                     # class labels (strings)
y2 = tabela["Fator"].astype(float).values    # regressão (numérico)

# X = todas as colunas exceto os alvos
X = tabela.drop(columns=["Conversao", "Fator"])

# Preencher NaNs numéricos nas features (imputação simples)
num_cols = X.select_dtypes(include=[np.number]).columns
if len(num_cols) > 0:
    X[num_cols] = X[num_cols].fillna(0)

# Se existirem colunas objeto restantes (não codificadas), transformá-las em str->encode ou preencher
obj_cols = X.select_dtypes(include=["object"]).columns.tolist()
if obj_cols:
    # força a transformar em string e mapear com LabelEncoder dinâmico (caso haja outros objeto)
    for c in obj_cols:
        X[c] = X[c].astype(str).fillna("Desconhecido")
        le = LabelEncoder()
        le.fit(X[c])
        X[c] = le.transform(X[c])
        codificadores[c] = le

# ---------- 4) Separar treino/teste ----------
X_treino, X_teste, y1_treino, y1_teste, y2_treino, y2_teste = train_test_split(
    X, y1, y2, test_size=0.3, random_state=42
)

# ---------- 5) Treinar modelos ----------
modelo_conversao = RandomForestClassifier(random_state=42)
modelo_fator = RandomForestRegressor(random_state=42)

modelo_conversao.fit(X_treino, y1_treino)
modelo_fator.fit(X_treino, y2_treino)

print("Acurácia (conversao):", modelo_conversao.score(X_teste, y1_teste))
print("R2 (fator):", modelo_fator.score(X_teste, y2_teste))

# ---------- 6) Ler novos fardos e aplicar transformações ----------
novos = safe_read_excel(ARQ_NOVOS)
novos = normalize_colnames(novos)

# Garantir colunas necessárias: se faltar alguma das features do X, cria com default
for c in X.columns:
    if c not in novos.columns:
        # se era coluna codificada, colocar -1 (unknown); se numérica, colocar 0
        if c in codificadores:
            novos[c] = -1
            print(f"Aviso: coluna '{c}' não estava em novos_fardos. Adicionada com -1.")
        else:
            novos[c] = 0
            print(f"Aviso: coluna numérica '{c}' não estava em novos_fardos. Adicionada com 0.")

# Aplicar codificadores treinados (com fallback -1)
for coluna, le in codificadores.items():
    if coluna in novos.columns:
        # força texto e preenche NaN, depois mapeia
        novos[coluna] = novos[coluna].astype(str).fillna("Desconhecido")
        # cria mapa rápido para membership test (as strings)
        classes = set([str(x) for x in le.classes_])
        def map_value(x):
            sx = str(x)
            if sx in classes:
                return int(le.transform([sx])[0])
            else:
                return -1
        novos[coluna] = novos[coluna].apply(map_value)

# Preencher NaN numéricos restantes
num_cols_novos = novos.select_dtypes(include=[np.number]).columns
if len(num_cols_novos) > 0:
    novos[num_cols_novos] = novos[num_cols_novos].fillna(0)

# Garantir mesma ordem de colunas/features
X_novos = novos[X.columns].copy()

# ---------- 7) Prever ----------
previsao_conversao = modelo_conversao.predict(X_novos)
previsao_fator = modelo_fator.predict(X_novos)

# ---------- 8) Montar resultado e reverter codificações ----------
resultado = novos.copy()
resultado["Conversao"] = previsao_conversao
resultado["Fator"] = previsao_fator

# Reverter codificadores para o texto original (tratando -1)
for coluna, le in codificadores.items():
    if coluna in resultado.columns:
        # converter com segurança para int, marcando inválidos
        vals_num = pd.to_numeric(resultado[coluna], errors="coerce")
        mask_validos = vals_num.notna() & (vals_num != -1)
        decoded = pd.Series(index=resultado.index, dtype=object)
        # aplicar inverse_transform apenas onde válido
        if mask_validos.any():
            arr = vals_num[mask_validos].astype(int).to_numpy()
            decoded_vals = le.inverse_transform(arr)
            decoded.loc[mask_validos] = decoded_vals
        # preencher desconhecidos
        decoded.loc[~mask_validos] = "Desconhecido"
        resultado[coluna] = decoded

# ---------- 9) Salvar ----------
resultado.to_excel(ARQ_SAIDA, index=False)
print("Resultado salvo em:", ARQ_SAIDA)
display(resultado)
